# Глава 14. MNIST: нейросеть узнаёт цифры

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь объяснить, что изображение для сети — это матрица чисел, обучить нейросеть на датасете MNIST и оценить её, а также привести пример, когда сеть уверенно ошибается.

Пришло время главного вау-момента фазы: мы обучим сеть распознавать рукописные цифры. Это классический «первый проект» в глубоком обучении — своего рода «Hello, world» нейросетей, только вместо надписи на экране сеть учится видеть. К концу главы ты напишешь цифру, а сеть её угадает.

## Изображение — это матрица чисел

**Аналогия: раскраска по клеточкам  💡**

Для нейросети картинка — не картинка, а таблица чисел. Чёрно-белое изображение цифры 28×28 пикселей — это сетка 28 на 28, где каждое число говорит, насколько клетка тёмная: 0 — белая, 255 — чёрная, промежуточные — оттенки серого. Сеть «видит» не «семёрку», а 784 числа (28 × 28). И именно в узоре этих чисел она находит закономерность, которая отличает семёрку от единицы.

Это ключевая идея всего компьютерного зрения: любое изображение — это просто числа. Цветная картинка — три таких сетки (красный, зелёный, синий канал). Как только картинка стала числами, с ней может работать нейросеть — точно так же, как с любыми другими числами.

## Датасет MNIST

MNIST — это знаменитый набор из 70 000 изображений рукописных цифр от 0 до 9, каждое размером 28×28 пикселей, собранных от тысяч разных людей. Он встроен прямо в Keras, качать вручную ничего не нужно — идеальный датасет для первого опыта с изображениями.

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt

# Загружаем: датасет сразу разделён на train и test
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Размер train:", X_train.shape)   # (60000, 28, 28)

# Посмотрим на первую цифру глазами
plt.imshow(X_train[0], cmap="gray")
plt.title(f"Это цифра: {y_train[0]}")
plt.show()

_Ожидаемый результат: картинка рукописной цифры с подписью её настоящего значения._

## Подготовка данных

Числа пикселей идут от 0 до 255. Сети удобнее работать с числами от 0 до 1 — это то самое масштабирование, что и в фазе 2, только здесь оно делается простым делением на 255. Масштабированные данные помогают сети учиться быстрее и стабильнее.

In [ ]:
# Масштабируем пиксели в диапазон 0..1
X_train = X_train / 255.0
X_test = X_test / 255.0

## Строим и обучаем сеть

Слой Flatten разворачивает картинку 28×28 в один длинный ряд из 784 чисел (сеть Dense принимает плоский список). Дальше — знакомые Dense-слои. Последний слой из 10 нейронов — по одному на каждую цифру от 0 до 9.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

model = Sequential([
    Flatten(input_shape=(28, 28)),        # картинка -> 784 числа
    Dense(128, activation="relu"),        # скрытый слой
    Dense(10, activation="softmax"),      # 10 цифр: вероятности
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Обучаем 5 эпох
model.fit(X_train, y_train, epochs=5)

# Проверяем на тесте, которого сеть не видела
loss, acc = model.evaluate(X_test, y_test)
print("Точность на тесте:", round(acc, 3))

_Ожидаемый результат: с каждой эпохой точность растёт; на тесте обычно около 0,97._

Обрати внимание на вывод во время обучения: после каждой эпохи Keras печатает текущую точность. Ты своими глазами видишь, как сеть «умнеет» с каждым проходом — от эпохи к эпохе ошибок всё меньше. Это та самая аналогия «ученик, исправляющий ошибки» в действии.

**Полезно знать  ℹ️**

softmax в последнем слое превращает выходы в вероятности по всем 10 цифрам, которые в сумме дают 1. Сеть не просто говорит «это 7», а сообщает: «на 95% это 7, на 3% это 1, на 2% остальное». Ответом становится цифра с самой высокой вероятностью. Это даёт бонус: можно узнать, насколько сеть уверена в своём ответе.

## Где сеть уверенно ошибается

Найдём картинки, на которых модель промахнулась, — это самое поучительное упражнение, важнее самой высокой точности.

In [ ]:
# Запускать в том же блокноте: используются обученная model и X_test,
# y_test из предыдущих ячеек этой главы (иначе NameError)
import numpy as np

# Предсказания на тесте
pred = model.predict(X_test)
pred_labels = np.argmax(pred, axis=1)   # цифра с макс. вероятностью

# Находим первую ошибку и смотрим на неё
wrong = np.where(pred_labels != y_test)[0][0]
plt.imshow(X_test[wrong], cmap="gray")
plt.title(f"Сеть: {pred_labels[wrong]}, правда: {y_test[wrong]}")
plt.show()

Посмотри на ошибочные картинки внимательно. Часто это криво написанные цифры, которые и человек угадал бы с трудом. Но иногда сеть промахивается на вполне читаемой цифре — и это важный урок о природе нейросетей.

**Это не магия  ⚠️**

Иногда сеть ошибается уверенно — и на кривой цифре, и даже на случайном шуме, где человек вообще не видит цифры, сеть может заявить «это точно 8 на 99%». Это ключевой урок: сеть не «понимает» цифры, как мы. Она нашла статистические узоры в числах пикселей. Там, где узор обманчив, она уверенно ошибётся. Нейросеть — мощнейший инструмент, но не разум, и слепо доверять её уверенности нельзя.

## Задания

Базовый уровень

Обучи сеть на MNIST и добейся точности выше 0,95 на тесте. Затем найди и покажи 3 картинки, на которых сеть ошиблась. Согласен ли ты с сетью, что они трудные, или ошибки странные?

Со звёздочкой  ⭐

Поэкспериментируй с архитектурой: добавь второй скрытый слой Dense(64) или увеличь число эпох до 15. Как меняется точность? Есть ли момент, когда точность на train растёт, а на test — нет? (Это переобучение из главы 7, теперь на нейросети.)

Вопросы для обсуждения  💬

Почему для нейросети картинка — это числа, а не «изображение»?

Что означает, что softmax даёт вероятности? Чем это полезнее простого ответа?

Если сеть уверенно ошибается на шуме, можно ли ей полностью доверять в важных задачах (медицина, беспилотники)?

**Проверь себя  ℹ️**

1. Как нейросеть «видит» картинку?

2. Зачем делить значения пикселей на 255?

3. Что делает слой softmax на выходе?